# Pooled post-Pre-Test physiology–MWL associations by group

This exploratory analysis uses all Test 1–3 and Evaluation MWL-aligned blocks, preserving repeated observations. Feature reduction is outcome-blind. Raw pooled Spearman correlations are followed by participant-cluster bootstrap confidence intervals and participant-label permutation tests for the descriptive Haptic–NoHA difference. A separate within-participant/within-phase centered sensitivity analysis addresses phase structure and repeated measurements.

The primary pooled p-values still do not model within-participant dependence and are not confirmatory. Evaluation is post-training without concurrent haptic feedback.

## Imports, configuration, and authoritative paths

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
pd.set_option("display.max_columns",120);pd.set_option("display.width",200)
RANDOM_SEED=20260902;HIGH_CORRELATION_THRESHOLD=.80;FDR_ALPHA=.05;N_BOOTSTRAP=2000;N_PERMUTATIONS=2000
INCLUDE_IMPUTED_MWL=True;SAVE_TABLES=True;SAVE_FIGURES=True
PHASES=["test_1","test_2","test_3","evaluation"];GROUPS=["Haptic","NoHA"];MODALITIES=["ECG","EDA","RESP","TEMP","fNIRS"]
def find_root(start=None):
 start=Path.cwd() if start is None else Path(start).resolve()
 for p in (start,*start.parents):
  if (p/"outputs/final_features/physiology_mwl_analysis_dataset.csv").exists():return p
 raise FileNotFoundError("repository root not found")
ROOT=find_root();INPUT=ROOT/"outputs/final_features/physiology_mwl_analysis_dataset.csv";TAB=ROOT/"postprocessing/outputs/tables";FIG=ROOT/"postprocessing/outputs/figures";TAB.mkdir(parents=True,exist_ok=True);FIG.mkdir(parents=True,exist_ok=True)
print("Input:",INPUT)

## Validate data and define the common outcome-blind reduced set

In [ ]:
all_data=pd.read_csv(INPUT);data=all_data[all_data.phase.isin(PHASES)].copy();data=data if INCLUDE_IMPUTED_MWL else data[data.mwl_source!="imputed_previous"].copy()
PREFIX={"ECG":"delta_ecg_","EDA":"delta_eda_","RESP":"delta_resp_","TEMP":"delta_temp_","fNIRS":"fnirs_"};modality_features={m:[c for c in all_data if c.startswith(p)] for m,p in PREFIX.items()};original=[f for m in MODALITIES for f in modality_features[m]];modality={f:m for m,fs in modality_features.items() for f in fs}
assert len(original)==79 and set(data.phase)==set(PHASES) and not data.duplicated(["participant_id","phase","block_index"]).any()
constants=[f for f in original if data[f].dropna().nunique()<2];expected=["delta_ecg_lf_hf_std","delta_ecg_phf_std","delta_ecg_plf_std"];assert sorted(constants)==sorted(expected);informative=[f for f in original if f not in constants]
nonmissing=data[original].notna().sum();missing=data[original].isna().mean();corr=data[informative].corr(method="spearman",min_periods=3);remaining=list(informative);removed={};step=0
while True:
 sub=corr.loc[remaining,remaining].abs();pairs=[]
 for i,a in enumerate(remaining):
  for b in remaining[i+1:]:
   v=sub.loc[a,b]
   if pd.notna(v) and v>=HIGH_CORRELATION_THRESHOLD:pairs.append((float(v),min(a,b),max(a,b)))
 if not pairs:break
 pair,a,b=sorted(pairs,key=lambda x:(-x[0],x[1],x[2]))[0];ma=data[a].isna().sum();mb=data[b].isna().sum();aa=float(sub.loc[a,[x for x in remaining if x!=a]].mean());bb=float(sub.loc[b,[x for x in remaining if x!=b]].mean())
 if ma!=mb:drop,keep=(a,b) if ma>mb else (b,a)
 elif not np.isclose(aa,bb,rtol=1e-12,atol=1e-12):drop,keep=(a,b) if aa>bb else (b,a)
 else:drop,keep=max(a,b),min(a,b)
 step+=1;removed[drop]=dict(removal_step=step,removed_because_correlated_with=keep,pair_abs_rho_at_removal=pair,mean_abs_rho_at_removal=aa if drop==a else bb,same_modality_as_representative=modality[drop]==modality[keep]);remaining.remove(drop)
retained=remaining;arr=corr.loc[retained,retained].abs().to_numpy(copy=True);np.fill_diagonal(arr,np.nan);max_remaining=float(np.nanmax(arr));assert max_remaining<HIGH_CORRELATION_THRESHOLD
audit=[]
for f in original:
 status="excluded_constant" if f in constants else ("excluded_high_correlation" if f in removed else "retained");d=removed.get(f,{})
 audit.append(dict(feature=f,modality=modality[f],n_nonmissing=int(nonmissing[f]),missing_fraction=missing[f],selection_status=status,retained=f in retained,removal_step=d.get("removal_step",np.nan),removed_because_correlated_with=d.get("removed_because_correlated_with",""),pair_abs_rho_at_removal=d.get("pair_abs_rho_at_removal",np.nan),mean_abs_rho_at_removal=d.get("mean_abs_rho_at_removal",np.nan),same_modality_as_representative=d.get("same_modality_as_representative",np.nan),selection_reason="constant/unusable" if f in constants else ("greater missingness, greater global redundancy, or deterministic tie-break within selected high-correlation pair" if f in removed else "retained after deterministic outcome-blind filtering")))
audit=pd.DataFrame(audit);reduced=pd.DataFrame({"feature":retained,"modality":[modality[f] for f in retained]})
if SAVE_TABLES:
 corr.to_csv(TAB/"physiological_feature_feature_spearman_matrix_no_pretest_080.csv",index_label="feature");audit.to_csv(TAB/"physiological_feature_correlation_filter_audit_no_pretest_080.csv",index=False);reduced.to_csv(TAB/"physiological_features_reduced_set_no_pretest_080.csv",index=False)
print("Rows",len(data),"participants",data.participant_id.nunique(),"original",len(original),"constants",len(constants),"high-correlation removals",len(removed),"retained",len(retained),"max remaining",max_remaining);print("Retained by modality:",reduced.groupby("modality").size().reindex(MODALITIES,fill_value=0).to_dict())

## Raw pooled group-specific Spearman correlations and separate group FDR

In [ ]:
def bh(p):
 p=np.asarray(p,float);out=np.full(p.shape,np.nan);mask=np.isfinite(p)
 if not mask.any():return out
 v=p[mask];order=np.argsort(v);ranked=v[order];n=len(v);q=np.minimum.accumulate((ranked*n/np.arange(1,n+1))[::-1])[::-1];restore=np.empty(n);restore[order]=np.clip(q,0,1);out[np.flatnonzero(mask)]=restore;return out
def corr_one(frame,f):
 d=frame[["participant_id","mwl_value",f]].dropna();base=dict(feature=f,modality=modality[f],n_observations=len(d),n_participants=d.participant_id.nunique(),n_unique_mwl=d.mwl_value.nunique())
 if len(d)<3 or d.participant_id.nunique()<2:return {**base,"rho":np.nan,"p_value":np.nan,"status":"insufficient_data"}
 if d.mwl_value.nunique()<2:return {**base,"rho":np.nan,"p_value":np.nan,"status":"constant_mwl"}
 if d[f].nunique()<2:return {**base,"rho":np.nan,"p_value":np.nan,"status":"constant_feature"}
 r=spearmanr(d[f],d.mwl_value);return {**base,"rho":float(r.statistic),"p_value":float(r.pvalue),"status":"ok"}
rows=[]
for group in GROUPS:
 r=pd.DataFrame([corr_one(data[data.group==group],f) for f in retained]);r.insert(0,"group",group);r["p_fdr"]=bh(r.p_value);r["significant_nominal"]=r.p_value.lt(FDR_ALPHA);r["significant_fdr"]=r.p_fdr.lt(FDR_ALPHA);rows.append(r)
raw=pd.concat(rows,ignore_index=True);raw=raw[["group","feature","modality","n_observations","n_participants","n_unique_mwl","rho","p_value","p_fdr","significant_nominal","significant_fdr","status"]]
if SAVE_TABLES:raw.to_csv(TAB/"physiology_mwl_group_correlations_no_pretest_reduced.csv",index=False)
print(raw.groupby("group").agg(valid=("status",lambda x:(x=="ok").sum()),nominal=("significant_nominal","sum"),fdr=("significant_fdr","sum")).to_string())

## Participant-cluster bootstrap and participant-label permutation comparison

Each bootstrap draw resamples participants within group and carries all their observations together. Each permutation reassigns complete participant clusters while preserving 11 Haptic and 10 NoHA labels. The two-sided permutation p-value compares absolute delta-rho to its participant-label null distribution.

In [ ]:
def rho_vector(frame):
 return frame[retained].corrwith(frame.mwl_value,axis=0,method="spearman").reindex(retained).to_numpy(float)
subjects={g:sorted(data.loc[data.group==g,"participant_id"].unique()) for g in GROUPS};by_subject={p:data[data.participant_id==p] for p in data.participant_id.unique()};rng=np.random.default_rng(RANDOM_SEED)
raw_wide=raw.pivot(index="feature",columns="group",values="rho").reindex(retained);observed_delta=(raw_wide.Haptic-raw_wide.NoHA).to_numpy(float)
boot=np.full((N_BOOTSTRAP,len(retained)),np.nan)
for i in range(N_BOOTSTRAP):
 pieces={g:pd.concat([by_subject[p] for p in rng.choice(subjects[g],size=len(subjects[g]),replace=True)],ignore_index=True) for g in GROUPS};boot[i]=rho_vector(pieces["Haptic"])-rho_vector(pieces["NoHA"])
ci_low=np.nanpercentile(boot,2.5,axis=0);ci_high=np.nanpercentile(boot,97.5,axis=0);boot_valid=np.isfinite(boot).sum(axis=0)
all_subjects=np.array(sorted(by_subject));n_h=len(subjects["Haptic"]);extreme=np.zeros(len(retained),int);perm_valid=np.zeros(len(retained),int)
for i in range(N_PERMUTATIONS):
 perm=rng.permutation(all_subjects);hset=set(perm[:n_h]);h=pd.concat([by_subject[p] for p in all_subjects if p in hset],ignore_index=True);n=pd.concat([by_subject[p] for p in all_subjects if p not in hset],ignore_index=True);delta=rho_vector(h)-rho_vector(n);valid=np.isfinite(delta)&np.isfinite(observed_delta);perm_valid+=valid;extreme+=valid&(np.abs(delta)>=np.abs(observed_delta))
perm_p=(extreme+1)/(perm_valid+1)
difference=pd.DataFrame({"feature":retained,"modality":[modality[f] for f in retained],"rho_haptic":raw_wide.Haptic.to_numpy(),"rho_noha":raw_wide.NoHA.to_numpy(),"delta_rho":observed_delta,"bootstrap_ci_low":ci_low,"bootstrap_ci_high":ci_high,"bootstrap_valid_replicates":boot_valid,"permutation_p_value":perm_p,"permutation_valid_replicates":perm_valid});difference["permutation_p_fdr"]=bh(difference.permutation_p_value);difference["difference_nominal"]=difference.permutation_p_value.lt(FDR_ALPHA);difference["difference_fdr"]=difference.permutation_p_fdr.lt(FDR_ALPHA)
if SAVE_TABLES:difference.to_csv(TAB/"physiology_mwl_haptic_noha_cluster_comparison_no_pretest.csv",index=False)
print("Bootstrap replicates",N_BOOTSTRAP,"permutations",N_PERMUTATIONS);print("Nominal differences",difference.difference_nominal.sum(),"FDR differences",difference.difference_fdr.sum())

## Phase-controlled within-participant sensitivity

For each feature, MWL and physiology are centered within participant × phase using pairwise-available rows. Only strata with at least two paired blocks contribute. Singleton strata—including Evaluation—contain no within-stratum information and are excluded. Correlations therefore describe whether block-to-block deviations in physiology covary with block-to-block MWL deviations, after removing participant-phase levels.

In [ ]:
controlled_rows=[]
for group in GROUPS:
 g=data[data.group==group]
 for f in retained:
  d=g[["participant_id","phase","mwl_value",f]].dropna().copy();d["pair_count"]=d.groupby(["participant_id","phase"])[f].transform("size");d=d[d.pair_count>=2];d["mwl_centered"]=d.mwl_value-d.groupby(["participant_id","phase"]).mwl_value.transform("mean");d["feature_centered"]=d[f]-d.groupby(["participant_id","phase"])[f].transform("mean")
  base=dict(group=group,feature=f,modality=modality[f],n_observations=len(d),n_participants=d.participant_id.nunique(),n_participant_phase_strata=d[["participant_id","phase"]].drop_duplicates().shape[0],n_unique_centered_mwl=d.mwl_centered.nunique())
  if len(d)<3 or d.mwl_centered.nunique()<2 or d.feature_centered.nunique()<2:r,p,status=np.nan,np.nan,"untestable_after_centering"
  else:q=spearmanr(d.feature_centered,d.mwl_centered);r,p,status=float(q.statistic),float(q.pvalue),"ok"
  controlled_rows.append({**base,"rho_phase_controlled":r,"p_value":p,"status":status})
controlled=pd.DataFrame(controlled_rows)
for group in GROUPS:
 mask=controlled.group==group;controlled.loc[mask,"p_fdr"]=bh(controlled.loc[mask,"p_value"])
controlled["significant_nominal"]=controlled.p_value.lt(FDR_ALPHA);controlled["significant_fdr"]=controlled.p_fdr.lt(FDR_ALPHA)
primary=raw[["group","feature","rho"]].rename(columns={"rho":"rho_raw_pooled"});controlled=controlled.merge(primary,on=["group","feature"],validate="one_to_one");controlled["rho_change_after_control"]=controlled.rho_phase_controlled-controlled.rho_raw_pooled
if SAVE_TABLES:controlled.to_csv(TAB/"physiology_mwl_phase_controlled_within_subject_correlations.csv",index=False)
print(controlled.groupby("group").agg(valid=("status",lambda x:(x=="ok").sum()),nominal=("significant_nominal","sum"),fdr=("significant_fdr","sum"),observations=("n_observations","max"),participants=("n_participants","max")).to_string())

## Publication-oriented group comparison figures

In [ ]:
colors={"ECG":"tab:red","EDA":"tab:green","RESP":"tab:blue","TEMP":"tab:orange","fNIRS":"tab:purple"}
fig,ax=plt.subplots(figsize=(7.5,7.5))
for m in MODALITIES:
 s=difference[difference.modality==m];ax.scatter(s.rho_noha,s.rho_haptic,label=m,color=colors[m],alpha=.8,s=45)
ax.axhline(0,color="grey",lw=1);ax.axvline(0,color="grey",lw=1);ax.plot([-1,1],[-1,1],"k--",lw=1)
for _,r in difference.nlargest(6,"delta_rho",keep="all").head(3).iterrows():ax.annotate(r.feature,(r.rho_noha,r.rho_haptic),xytext=(4,4),textcoords="offset points",fontsize=7)
for _,r in difference.nsmallest(3,"delta_rho").iterrows():ax.annotate(r.feature,(r.rho_noha,r.rho_haptic),xytext=(4,4),textcoords="offset points",fontsize=7)
ax.set(xlim=(-1,1),ylim=(-1,1),xlabel="Spearman rho: NoHA",ylabel="Spearman rho: Haptic",title="Pooled post-Pre-Test physiology–MWL associations");ax.set_aspect("equal");ax.legend();ax.grid(alpha=.2);fig.tight_layout()
if SAVE_FIGURES:fig.savefig(FIG/"physiology_mwl_haptic_vs_noha_no_pretest_reduced.png",dpi=200,bbox_inches="tight")
plt.show()
plot=difference.sort_values("delta_rho").reset_index(drop=True);y=np.arange(len(plot));fig,ax=plt.subplots(figsize=(10,max(10,len(plot)*.26)));err=np.vstack([plot.delta_rho-plot.bootstrap_ci_low,plot.bootstrap_ci_high-plot.delta_rho]);ax.errorbar(plot.delta_rho,y,xerr=err,fmt="o",ms=4,color="black",ecolor="grey",capsize=2);ax.axvline(0,color="black",ls="--",lw=1);ax.set_yticks(y,plot.feature,fontsize=6);ax.set(xlabel="rho_Haptic − rho_NoHA (95% participant-cluster bootstrap CI)",title="Participant-level comparison of pooled physiology–MWL correlations");ax.grid(axis="x",alpha=.2);fig.tight_layout()
if SAVE_FIGURES:fig.savefig(FIG/"physiology_mwl_haptic_noha_delta_rho_cluster_ci.png",dpi=200,bbox_inches="tight")
plt.show()

## Concise reproducible summary

In [ ]:
print("ANALYSIS SUMMARY");print("Phases:",PHASES,"observations",len(data),"participants",data.participant_id.nunique());print("Features: original",len(original),"constant",len(constants),"redundant removed",len(removed),"retained",len(retained));print("Retained by modality",reduced.groupby("modality").size().reindex(MODALITIES,fill_value=0).to_dict());print("Maximum remaining feature |rho|",max_remaining)
for g in GROUPS:
 s=raw[raw.group==g];top=s.assign(a=s.rho.abs()).nlargest(5,"a");print(f"{g}: N observations={s.n_observations.max()}, subjects={s.n_participants.max()}, nominal={s.significant_nominal.sum()}, FDR={s.significant_fdr.sum()}; strongest "+"; ".join(f"{r.feature} ({r.rho:+.3f})" for _,r in top.iterrows()))
print("Largest descriptive/inferential delta-rho results:\n",difference.nlargest(10,"delta_rho",keep="all")[["feature","modality","rho_haptic","rho_noha","delta_rho","bootstrap_ci_low","bootstrap_ci_high","permutation_p_value","permutation_p_fdr"]].head(10).to_string(index=False));print("Phase-controlled sensitivity:\n",controlled.groupby("group").agg(valid=("status",lambda x:(x=="ok").sum()),nominal=("significant_nominal","sum"),fdr=("significant_fdr","sum")).to_string());print("Primary pooled associations preserve repeated blocks and remain exploratory. Feature selection used no MWL or group-association result.")